# <span style="color: #0099ff; display: block; text-align: center;">CREAR CATALOGO</span>

- Crear catalog y schemas (bases de datos) para las capas: bronze, silver y gold.
- Para la zona de aterrizaje no sera creada una base de datos sino un volumen dentro del schema bronze.

In [0]:
# 1. Parametros
dbutils.widgets.removeAll()

dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "prod"], "Ambiente")
environment = dbutils.widgets.get("environment")

dbutils.widgets.text("catalog", "Medallion", "Catalogo")
catalogName = dbutils.widgets.get("catalog") + "_" + environment.lower()
display(catalogName)

'Medallion_dev'

In [0]:
try: # 2. Crear catalogo
    if not any(catalog.name == catalogName for catalog in spark.catalog.listCatalogs()):
        spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalogName}` COMMENT 'Catalogo demo arquitectura Medallion'");

    spark.sql(f"USE CATALOG {catalogName}")
    print(f"Catalgo actual: {spark.catalog.currentCatalog()}")
except Exception as e:
    msg = f"Error al crear o seleccionar catalogo: {catalogName}. Mensaje: {e}"
    raise Exception(msg)    

Catalgo actual: medallion_dev


In [0]:
try: # 3. Crear bases de datos
    schemaNames = ["bronze", "silver", "gold"]
    for schemaName in schemaNames:            
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalogName}`.`{schemaName}` COMMENT 'Schema para capa de datos {schemaName}'")
        print(f"Schema: `{schemaName}`")
except Exception as e:
    raise Exception(f"Error al crear esquemas en el catalogo: {catalogName}. Mensaje: {e}")

Schema: `bronze`
Schema: `silver`
Schema: `gold`


In [0]:
schema_bronze = "bronze"
volume_name = "landing_raw"
volume_folder =  f"/Volumes/{catalogName}/{schema_bronze}/{volume_name}"

try: # 4. Crear volumen
    print(f"Usando `{catalogName}`.`{schema_bronze}`")
    spark.sql(f"""USE `{catalogName}`.`{schema_bronze}`""")    
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalogName}`.`{schema_bronze}`.`{volume_name}` COMMENT 'Zona de aterrizaje para datos brutos (raw)'")
    print(f"Volumen creado o existente: {catalogName}/{schema_bronze}/{volume_name}")
except Exception as e:
    raise Exception(f"Error al crear volumen: {catalogName}/{schema_bronze}/{volume_name}. Mensaje: {e}")

Usando `Medallion_dev`.`bronze`
Volumen creado o existente: Medallion_dev/bronze/landing_raw


In [0]:
print(f"Contenido del volumen: {volume_folder}")
fileinfo = dbutils.fs.ls(volume_folder)
directories = [d[1] for d in fileinfo]
print(directories)
#display(fileinfo)

Contenido del volumen: /Volumes/Medallion_dev/bronze/landing_raw
[]
